In [1]:
"""
MetaData:
---------
Developer: Swapnendu Banik
Description: DQN implementation for CartPole-v1 environment.
Problem Statement: Train an agent to balance a pole on a cart using Deep Q-Network (DQN).
Recommended IDE: Google Colab

Recommended Videos:
-------------------
1. Understanding DQN [Theory]: https://youtu.be/x83WmvbRa2I?si=qUBOc9s--_-TRxOb
2. Understanding SeLU Activation Function Used in Model Arch: https://youtu.be/2OwWs7Hzr9g?si=RlLPiwLWN0pITEVD
3. Code A DQN-Part1 (SentDex) {Diff Arch, Diff Approach, Good Understanding}: https://youtu.be/t3fbETsIBCY?si=YVRa-qdPH2Ayqs-Z
4. Code A DQN-Part2 (SentDex) {Diff Arch, Diff Approach, Good Understanding}: https://youtu.be/qfovbG84EBg?si=zjTuFFVSae5mDIsw

Operations Carried Out:
------------------------
- Environment setup (OpenAI Gym CartPole-v1)
- Model definition (3-layer fully connected neural network)
- DQN training loop (epsilon-greedy exploration, experience replay)
- Periodic testing and model saving
- Evaluation of trained model performance

Hyperparameters:
----------------
- learning_rate: 1e-4
- gamma: 0.99
- epsilon: 1
- min_epsilon: 0.1
- epsilon_decay: 0.9 / 2.5e3
- memory_len: 10000
- target_update_delay: 2
- test_delay: 10
- episode_limit: 100
- train_batch_size: 100

Environment: CartPole-v1
Algorithm: Deep Q-Network (DQN)
Library: ["PyTorch", "OpenAI-Gym"]
"""

'\nMetaData:\n---------\nDeveloper: Swapnendu Banik\nDescription: DQN implementation for CartPole-v1 environment.\nProblem Statement: Train an agent to balance a pole on a cart using Deep Q-Network (DQN).\nRecommended IDE: Google Colab\n\nRecommended Videos:\n-------------------\n1. Understanding DQN [Theory]: https://youtu.be/x83WmvbRa2I?si=qUBOc9s--_-TRxOb\n2. Understanding SeLU Activation Function Used in Model Arch: https://youtu.be/2OwWs7Hzr9g?si=RlLPiwLWN0pITEVD\n3. Code A DQN-Part1 (SentDex) {Diff Arch, Diff Approach, Good Understanding}: https://youtu.be/t3fbETsIBCY?si=YVRa-qdPH2Ayqs-Z\n4. Code A DQN-Part2 (SentDex) {Diff Arch, Diff Approach, Good Understanding}: https://youtu.be/qfovbG84EBg?si=zjTuFFVSae5mDIsw\n\nOperations Carried Out:\n------------------------\n- Environment setup (OpenAI Gym CartPole-v1)\n- Model definition (3-layer fully connected neural network)\n- DQN training loop (epsilon-greedy exploration, experience replay)\n- Periodic testing and model saving\n- Ev

In [6]:
import warnings

# Filter the specific warning
warnings.filterwarnings("ignore", category=DeprecationWarning, message="`np.bool8` is a deprecated alias")


## Model Architecture for DQN

In [2]:
from torch import nn
from torch.nn import functional


class Model(nn.Module):
    """
    Defines a fully connected neural network model for Q-learning with three layers.

    Args:
    - input_features (int): Number of input features (state dimension).
    - output_values (int): Number of output values (action space size).
    """
    def __init__(self, input_features, output_values):
        super(Model, self).__init__()  # Initialize the parent class (nn.Module)

        # First fully connected layer (input layer to hidden layer)
        self.fc1 = nn.Linear(in_features=input_features, out_features=32)

        # Second fully connected layer (hidden layer to another hidden layer)
        self.fc2 = nn.Linear(in_features=32, out_features=32)

        # Third fully connected layer (hidden layer to output layer)
        self.fc3 = nn.Linear(in_features=32, out_features=output_values)

    def forward(self, x):
        """
        Defines the forward pass of the neural network. Applies SELU activation
        on the first two layers, and returns the output from the last layer.

        Args:
        - x (tensor): The input tensor (state).

        Returns:
        - tensor: Output Q-values (predicted action values).
        """
        x = functional.selu(self.fc1(x))  # Apply SELU activation to first layer
        x = functional.selu(self.fc2(x))  # Apply SELU activation to second layer
        x = self.fc3(x)  # No activation after the last layer (output layer)
        return x


#### Import necessary libraries

In [3]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import gym
from collections import deque
import random

## DQN training on 'CartPole-v1'

In [4]:
# Parameters
use_cuda = True  # Use GPU if available for faster computation
episode_limit = 100  # Maximum number of episodes for training
target_update_delay = 2  # Update the target network every 2 episodes
test_delay = 10  # Test the model's performance every 10 episodes
learning_rate = 1e-4  # Learning rate for the optimizer
epsilon = 1  # Initial exploration rate for the epsilon-greedy policy
min_epsilon = 0.1  # Minimum value for epsilon to ensure some exploration
epsilon_decay = 0.9 / 2.5e3  # Rate at which epsilon decays over time
gamma = 0.99  # Discount factor for future rewards
memory_len = 10000  # Maximum length of the replay memory buffer

# Initialize the CartPole-v1 environment from OpenAI Gym
env = gym.make('CartPole-v1')

# Get the number of features (observations) from the environment's state space
n_features = len(env.observation_space.high)

# Get the number of possible actions from the environment's action space
n_actions = env.action_space.n

# Initialize the replay memory with a fixed size
memory = deque(maxlen=memory_len)
# Each memory entry is stored as a tuple: (state, action, reward, next_state)

# Set the device for computations (GPU if available, else CPU)
device = torch.device("cuda" if use_cuda and torch.cuda.is_available() else "cpu")

# Define the loss function for the Q-learning updates
criterion = nn.MSELoss()

# Initialize the policy network that will be used to make predictions for actions
policy_net = Model(n_features, n_actions).to(device)

# Initialize the target network that will stabilize training by periodically syncing weights from the policy network
target_net = Model(n_features, n_actions).to(device)

# Copy the initial weights of the policy network to the target network
target_net.load_state_dict(policy_net.state_dict())

# Set the target network to evaluation mode since it does not need to be trained
target_net.eval()




def get_states_tensor(sample, states_idx):
    """
    Converts a list of state samples into a PyTorch tensor for batch processing.

    Args:
    sample (list): A list of samples, where each sample contains states, actions, and other information.
    states_idx (int): The index in the sample tuple where the state information is stored.

    Returns:
    torch.Tensor: A tensor of shape (sample_len, n_features), where each row represents a state.
    """
    sample_len = len(sample)  # Number of samples
    states_tensor = torch.empty((sample_len, n_features), dtype=torch.float32, requires_grad=False)
    # Initialize an empty tensor to store the states

    features_range = range(n_features)  # Range for the number of features in a state
    for i in range(sample_len):  # Iterate through each sample
        for j in features_range:  # Iterate through each feature of the state
            states_tensor[i, j] = sample[i][states_idx][j].item()
            # Extract the feature value from the sample and assign it to the tensor

    return states_tensor  # Return the resulting tensor


def normalize_state(state):
    """
    Normalizes the state values to bring them to a comparable scale.

    Args:
    state (list): A list representing the state with 4 features.

    Modifies:
    state (list): Normalizes the features in-place.
    """
    state[0] /= 2.5  # Scale the first feature
    state[1] /= 2.5  # Scale the second feature
    state[2] /= 0.3  # Scale the third feature
    state[3] /= 0.3  # Scale the fourth feature


def state_reward(state, env_reward):
    """
    Modifies the environment reward by penalizing deviations in position and angle.

    Args:
    state (list): A list representing the current state with 4 features.
    env_reward (float): The raw reward from the environment.

    Returns:
    float: The adjusted reward, penalizing deviation in position and angle.
    """
    return env_reward - (abs(state[0]) + abs(state[2])) / 2.5
    # Subtracts a penalty based on the absolute position (state[0]) and angle (state[2])
    # Reducing the reward for being far from the ideal state.



def get_action(state, e=min_epsilon):
    """
    Determines the action to take (exploration or exploitation) based on the epsilon value.

    Args:
    state (list or np.array): The current state of the environment.
    e (float): The epsilon value controlling the exploration-exploitation trade-off.

    Returns:
    int: The action to take (an integer representing the action index).
    """
    if random.random() < e:
        # Exploration: Choose a random action with probability `e`.
        action = random.randrange(0, n_actions)
    else:
        # Exploitation: Use the policy network to predict the best action.
        state = torch.tensor(state, dtype=torch.float32, device=device)  # Convert state to a PyTorch tensor
        action = policy_net(state).argmax().item()  # Predict the action with the highest Q-value

    return action  # Return the chosen action


def fit(model, inputs, labels):
    """
    Trains the model on a batch of data using Mean Squared Error loss.

    Args:
    model (torch.nn.Module): The neural network model to train.
    inputs (torch.Tensor): The input data (states).
    labels (torch.Tensor): The target Q-values for the given states.

    Returns:
    float: The average loss over the entire dataset.
    """
    inputs = inputs.to(device)  # Move inputs to the appropriate device (CPU or GPU)
    labels = labels.to(device)  # Move labels to the appropriate device (CPU or GPU)
    train_ds = TensorDataset(inputs, labels)  # Create a dataset from the inputs and labels
    train_dl = DataLoader(train_ds, batch_size=5)  # Create a data loader with a batch size of 5

    optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate)  # Define the optimizer
    model.train()  # Set the model to training mode
    total_loss = 0.0  # Initialize total loss

    for x, y in train_dl:
        # Iterate through the training data in batches
        out = model(x)  # Forward pass: Get predictions from the model
        loss = criterion(out, y)  # Compute the loss (Mean Squared Error)
        total_loss += loss.item()  # Accumulate the loss for reporting
        optimizer.zero_grad()  # Reset gradients to zero
        loss.backward()  # Backward pass: Compute gradients
        optimizer.step()  # Update model parameters using the optimizer

    model.eval()  # Set the model to evaluation mode

    return total_loss / len(inputs)  # Return the average loss over all inputs



def optimize_model(train_batch_size=100):
    """
    Optimizes the policy network using a batch of experiences from memory.

    Args:
    train_batch_size (int): Number of experiences to sample from memory for training.
    """
    train_batch_size = min(train_batch_size, len(memory))  # Ensure batch size doesn't exceed memory size
    train_sample = random.sample(memory, train_batch_size)  # Randomly sample a batch of experiences from memory

    state = get_states_tensor(train_sample, 0)  # Extract current states as tensors
    next_state = get_states_tensor(train_sample, 3)  # Extract next states as tensors

    q_estimates = policy_net(state.to(device)).detach()  # Predicted Q-values for current states
    next_state_q_estimates = target_net(next_state.to(device)).detach()  # Predicted Q-values for next states

    for i in range(len(train_sample)):
        # Update Q-value for the chosen action using the Bellman equation
        q_estimates[i][train_sample[i][1]] = (
            state_reward(next_state[i], train_sample[i][2]) + gamma * next_state_q_estimates[i].max()
        )

    fit(policy_net, state, q_estimates)  # Train the policy network with updated Q-values



def train_one_episode():
    """
    Trains the model for one episode by interacting with the environment.
    Uses epsilon-greedy exploration and updates the memory and model.

    Returns:
    score (float): Total environment reward received during the episode.
    reward (float): Total modified reward calculated using `state_reward`.
    """
    global epsilon  # Access the global epsilon for exploration-exploitation
    current_state = env.reset()  # Reset the environment and get the initial state
    normalize_state(current_state)  # Normalize the state for consistency
    done = False  # Initialize the termination flag
    score = 0  # Initialize the environment reward tracker
    reward = 0  # Initialize the modified reward tracker

    while not done:
        action = get_action(current_state, epsilon)  # Choose an action using epsilon-greedy strategy
        next_state, env_reward, done, _ = env.step(action)  # Perform the action in the environment
        normalize_state(next_state)  # Normalize the resulting state
        memory.append((current_state, action, env_reward, next_state))  # Store experience in memory
        current_state = next_state  # Update the current state
        score += env_reward  # Accumulate environment reward
        reward += state_reward(next_state, env_reward)  # Accumulate modified reward

        optimize_model(100)  # Optimize the policy network after every step

        epsilon -= epsilon_decay  # Decay epsilon for less exploration over time

    return score, reward  # Return the rewards for the episode



def test():
    """
    Tests the current policy network by running one episode without exploration.

    Returns:
    score (float): Total environment reward during the test episode.
    reward (float): Total modified reward calculated using `state_reward`.
    """
    state = env.reset()  # Reset the environment for testing
    normalize_state(state)  # Normalize the initial state
    done = False  # Initialize the termination flag
    score = 0  # Initialize the environment reward tracker
    reward = 0  # Initialize the modified reward tracker

    while not done:
        action = get_action(state)  # Choose the best action without exploration
        state, env_reward, done, _ = env.step(action)  # Perform the action in the environment
        normalize_state(state)  # Normalize the resulting state
        score += env_reward  # Accumulate environment reward
        reward += state_reward(state, env_reward)  # Accumulate modified reward

    return score, reward  # Return the rewards for the test episode



def main():
    """
    Main function to train and test the model over multiple episodes.
    Includes periodic testing and saving the best-performing model.
    """
    best_test_reward = 0  # Initialize the best test reward

    for i in range(episode_limit):  # Loop through the episodes
        score, reward = train_one_episode()  # Train the model for one episode

        print(f'Episode {i + 1}: score: {score} - reward: {reward}')  # Log the results

        if i % target_update_delay == 0:
            # Update the target network periodically
            target_net.load_state_dict(policy_net.state_dict())
            target_net.eval()

        if (i + 1) % test_delay == 0:
            # Test the model periodically
            test_score, test_reward = test()
            print(f'Test Episode {i + 1}: test score: {test_score} - test reward: {test_reward}')
            if test_reward > best_test_reward:
                # Save the model if it achieves a new best test reward
                print('New best test reward. Saving model')
                best_test_reward = test_reward
                torch.save(policy_net.state_dict(), 'policy_net.pth')

    if episode_limit % test_delay != 0:
        # Test the model after the final episode if not already tested
        test_score, test_reward = test()
        print(f'Test Episode {episode_limit}: test score: {test_score} - test reward: {test_reward}')
        if test_reward > best_test_reward:
            print('New best test reward. Saving model')
            best_test_reward = test_reward
            torch.save(policy_net.state_dict(), 'policy_net.pth')

    print(f'best test reward: {best_test_reward}')  # Log the best test reward


if __name__ == '__main__':
    main()  # Run the main function


Episode 1: score: 35.0 - reward: 30.999055713415142
Episode 2: score: 15.0 - reward: 13.000594276189803
Episode 3: score: 10.0 - reward: 8.648637077212333
Episode 4: score: 51.0 - reward: 46.254001747071726
Episode 5: score: 33.0 - reward: 28.8776682086289
Episode 6: score: 23.0 - reward: 20.32912156879902
Episode 7: score: 31.0 - reward: 27.650117721408606
Episode 8: score: 18.0 - reward: 16.064629155769946
Episode 9: score: 29.0 - reward: 26.41876246333122
Episode 10: score: 23.0 - reward: 18.934342367760838
Test Episode 10: test score: 23.0 - test reward: 19.218327596783638
New best test reward. Saving model
Episode 11: score: 25.0 - reward: 21.942147903889417
Episode 12: score: 13.0 - reward: 11.5552026681602
Episode 13: score: 17.0 - reward: 15.269554605707528
Episode 14: score: 30.0 - reward: 26.809396826103328
Episode 15: score: 15.0 - reward: 12.88317917585373
Episode 16: score: 20.0 - reward: 16.88214417994022
Episode 17: score: 35.0 - reward: 31.067877155542384
Episode 18: sc

## Testing Out Model Performance

In [15]:
use_cuda=True
device = torch.device("cuda" if use_cuda and torch.cuda.is_available() else "cpu")
test_policy_net = Model(4, 2).to(device)  # Adjust input/output dimensions if needed
trained_model_path="trained_weights\policy_net.pth"
test_policy_net.load_state_dict(torch.load(trained_model_path, map_location=device))
test_policy_net.eval()  # Set the model to evaluation mode

# Create the environment
env = gym.make('CartPole-v1')

def normalize_state(state):
    state[0] /= 2.5
    state[1] /= 2.5
    state[2] /= 0.3
    state[3] /= 0.3

# Number of episodes to evaluate
num_episodes = 10  # You can adjust this as needed

total_rewards = []

for episode in range(num_episodes):
    state = env.reset()
    done = False
    episode_reward = 0

    while not done:
        # Get action from the loaded policy network
        state_tensor = torch.tensor(state, dtype=torch.float32, device=device)
        action = test_policy_net(state_tensor).argmax().item()

        # Take action in the environment
        state, reward, done, _ = env.step(action) ## Use if not using new_step_api
        # state, reward, terminated, truncated, _ = env.step(action) if using new_step_api
        episode_reward += reward

    print(f"Episode {episode} Reward: {episode_reward}")

    total_rewards.append(episode_reward)

# Calculate average reward
avg_reward = sum(total_rewards) / num_episodes
print(f"Average Reward over {num_episodes} episodes: {avg_reward}")

env.close()  # Close the environment when done

<ipython-input-15-1b12eba4a0cb>:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  test_policy_net.load_state_dict(torch.load(trained_model_path, map_location=device))


Episode 0 Reward: 500.0
Episode 1 Reward: 500.0
Episode 2 Reward: 500.0
Episode 3 Reward: 500.0
Episode 4 Reward: 500.0
Episode 5 Reward: 500.0
Episode 6 Reward: 500.0
Episode 7 Reward: 500.0
Episode 8 Reward: 500.0
Episode 9 Reward: 500.0
Average Reward over 10 episodes: 500.0
